In [ ]:
"""
Week 1: IoT Telemetry Ingestion and Signal Processing

Author: Subhashree Behera

Objective:
Create rolling-window statistical features from industrial sensor data
for predictive maintenance analysis.

Steps Performed:

1. Load fused AI4I dataset.
2. Sort records chronologically by date.
3. Select machine sensor columns.
4. Apply rolling window (window = 5).
5. Compute:

   * Rolling Mean
   * Rolling Standard Deviation
   * Rolling Variance
6. Generate 15 new engineered features.
7. Save the processed dataset for downstream analysis.

Output:
Feature-engineered dataset containing rolling statistics for:

* Air Temperature
* Process Temperature
* Rotational Speed
* Torque
* Tool Wear

Purpose:
Capture short-term operational trends and variability that may
indicate machine degradation or impending failure.
"""


In [1]:
# Step 1: Import required libraries
# pandas -> data manipulation, numpy -> numerical operations
import pandas as pd
import numpy as np

In [2]:
# Step 2: Load the fused dataset (sensors + weather, merged by date)
df = pd.read_csv('../data/ai4i_fused.csv')

print("Shape:", df.shape)
print("Columns:", list(df.columns))
df.head()

Shape: (10000, 21)
Columns: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'date', 'avg_temp_c', 'min_temp_c', 'max_temp_c', 'precipitation_mm', 'avg_wind_speed_kmh', 'avg_sea_level_pres_hpa']


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,...,PWF,OSF,RNF,date,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,avg_wind_speed_kmh,avg_sea_level_pres_hpa
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,...,0,0,0,2020-01-01,25.7,22.5,30.3,27.9,6.4,1014.4


In [3]:
# Step 3: Strip any accidental leading/trailing spaces from column names
# This prevents bugs later when referencing columns by exact name
df.columns = df.columns.str.strip()

print("Columns cleaned. Sample:", list(df.columns[:5]))

Columns cleaned. Sample: ['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]']


In [50]:
df = df.sort_values('date').reset_index(drop=True)

In [51]:
# Sensor columns to apply rolling window on
sensors = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

In [52]:
window = 5 # rolling window size

for col in sensors:
    df[f'{col}_rolling_mean'] = (
        df[col]
        .rolling(window=window, min_periods=1)
        .mean()
    )

    df[f'{col}_rolling_std'] = (
        df[col]
        .rolling(window=window, min_periods=1)
        .std()
        .fillna(0)
    )

    df[f'{col}_rolling_var'] = (
        df[col]
        .rolling(window=window, min_periods=1)
        .var()
        .fillna(0)
    )

In [53]:
print("Shape before:", (10000, 21))
print("Shape after:", df.shape)
print(df.tail(10))

Shape before: (10000, 21)
Shape after: (10000, 36)
        UDI Product ID Type  Air temperature [K]  Process temperature [K]  \
9990   9991     L57170    L                298.8                    308.5   
9991   9992     M24851    M                298.9                    308.4   
9992   9993     L57172    L                298.8                    308.4   
9993   9994     L57173    L                298.8                    308.4   
9994   9995     L57174    L                298.8                    308.3   
9995   9996     M24855    M                298.8                    308.4   
9996   9997     H39410    H                298.9                    308.4   
9997   9998     M24857    M                299.0                    308.6   
9998   9999     H39412    H                299.0                    308.7   
9999  10000     M24859    M                299.0                    308.7   

      Rotational speed [rpm]  Torque [Nm]  Tool wear [min]  Machine failure  \
9990                  

In [54]:
df.to_csv('../data/ai4i_rolling_features.csv', index=False)